In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
%pip install kaggle huggingface-hub tqdm matplotlib scikit-learn Pillow

In [ ]:
import os
from google.colab import files

print("Upload your kaggle.json API key:")
files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle API key configured.")

In [ ]:
import kaggle

kaggle.api.dataset_download_files(
    "paultimothymooney/chest-xray-pneumonia",
    path="/content/data",
    unzip=True
)

print("Dataset downloaded.")

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

REPO_PATH = Path("/content/drive/MyDrive/adversarial-ai-attacks-mitigations")
DATA_DIR  = Path("/content/data/chest_xray")
MODEL_DIR = REPO_PATH / "shared" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir : {DATA_DIR}")
print(f"Model dir: {MODEL_DIR}")

In [ ]:
import sys
sys.path.append(str(REPO_PATH / "ch01_ml_foundations"))

from dataset import verify_dataset_structure, count_images, report_dataset_stats, get_dataloaders
from model import build_model, get_device, unfreeze_backbone
from train import train
from evaluate import run_full_evaluation

In [ ]:
is_valid = verify_dataset_structure(DATA_DIR)
if not is_valid:
    raise RuntimeError("Dataset structure invalid. Check download.")

counts = count_images(DATA_DIR)
report_dataset_stats(counts)

In [ ]:
device = get_device()
model = build_model(num_classes=2, dropout=0.3)
model = model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters   : {total - trainable:,}")

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=64,
    num_workers=4
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

In [ ]:
model = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=10,
    learning_rate=1e-3,
    checkpoint_dir=MODEL_DIR
)

In [ ]:
EVAL_DIR = REPO_PATH / "ch01_ml_foundations" / "eval_outputs"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

metrics = run_full_evaluation(
    model=model,
    loader=test_loader,
    device=device,
    save_dir=EVAL_DIR
)

print(f"\nFinal Test Accuracy : {metrics['accuracy']:.4f}")
print(f"Final Test AUC      : {metrics['auc']:.4f}")

In [ ]:
from huggingface_hub import HfApi, login
import shutil

print("Enter your HuggingFace token:")
login()

api = HfApi()
repo_id = "theinferenceloop/adversarial-ai-target"

api.create_repo(repo_id=repo_id, exist_ok=True)

api.upload_file(
    path_or_fileobj=str(MODEL_DIR / "best_checkpoint.pt"),
    path_in_repo="best_checkpoint.pt",
    repo_id=repo_id
)

print(f"Model pushed to HuggingFace: {repo_id}")

In [ ]:
import shutil

dest = REPO_PATH / "ch01_ml_foundations" / "eval_outputs"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(EVAL_DIR, dest)

print(f"Eval outputs saved to repo: {dest}")
print("Done. Sync your Google Drive folder to pull results locally.")

In [ ]:
print("\n=== Model Card Summary ===")
print(f"Model      : EfficientNet-B3")
print(f"Task       : Binary chest X-ray classification")
print(f"Classes    : NORMAL (0), PNEUMONIA (1)")
print(f"Dataset    : Kaggle chest-xray-pneumonia (5,863 images)")
print(f"Input size : 300x300 RGB")
print(f"Accuracy   : {metrics['accuracy']:.4f}")
print(f"AUC        : {metrics['auc']:.4f}")
print(f"HuggingFace: theinferenceloop/adversarial-ai-target")
print(f"Attack surface: ch04-ch09, ch12")